# MLForecast Ensemble Training
Reads training data from BigQuery, fits XGB / LGBM / Ridge / RF / ET models via MLForecast,
evaluates each on a hold-out set, selects the champion by WMAPE, writes predictions and
champion metadata back to BigQuery, and saves the champion model locally.

In [ ]:
import json
import logging
import os
from pathlib import Path

import numpy as np
import pandas as pd
from google.cloud import bigquery
from lightgbm import LGBMRegressor
from mlforecast import MLForecast
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
logger = logging.getLogger("main")

## Configuration
Edit these variables before running the notebook.

In [ ]:
PROJECT_ID           = "dazzling-seat-366014"
DATASET_ID           = "forecasting"
TABLE_NAME           = "sales_daily"
BQ_TABLE             = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME}"
PREDICTION_BQ_TABLE  = f"{PROJECT_ID}.{DATASET_ID}.sales_predictions"
CHAMPION_BQ_TABLE    = f"{PROJECT_ID}.{DATASET_ID}.sales_champion"
MODEL_DISPLAY_NAME   = "sales-mlforecast-v3"
MODEL_OUTPUT_DIR     = Path(os.environ.get("AIP_MODEL_DIR", "model_output"))

FORECAST_FREQ   = "D"
HORIZON         = 2
LAGS            = [1, 2, 3]
DATE_FEATURES   = ["dayofweek", "month"]
MODEL_TYPES     = ["lgbm", "xgb", "rf", "et", "ridge"]
CHAMPION_METRIC = "wmape"

## Helper Functions

In [ ]:
def wmape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    denom = np.abs(y_true).sum()
    if denom == 0:
        return float("nan")
    return float(np.abs(y_true - y_pred).sum() / denom)


def build_model(model_type: str):
    if model_type == "lgbm":
        return LGBMRegressor(
            n_estimators=300, learning_rate=0.05, num_leaves=64,
            subsample=0.9, colsample_bytree=0.9, random_state=42, verbosity=-1,
        )
    if model_type == "xgb":
        return XGBRegressor(
            n_estimators=300, learning_rate=0.05, max_depth=6,
            subsample=0.9, colsample_bytree=0.9, random_state=42,
            eval_metric="rmse", verbosity=0,
        )
    if model_type == "rf":
        return RandomForestRegressor(
            n_estimators=300, max_depth=10, min_samples_leaf=2,
            n_jobs=-1, random_state=42,
        )
    if model_type == "et":
        return ExtraTreesRegressor(
            n_estimators=300, max_depth=10, min_samples_leaf=2,
            n_jobs=-1, random_state=42,
        )
    if model_type == "ridge":
        return Ridge(alpha=1.0)
    raise ValueError(f"Unsupported model_type: {model_type}")

## 1. Load Data from BigQuery

In [ ]:
client = bigquery.Client(project=PROJECT_ID)

query = f"""
    SELECT unique_id, ds, y
    FROM `{BQ_TABLE}`
    WHERE unique_id IS NOT NULL
      AND ds IS NOT NULL
      AND y IS NOT NULL
    ORDER BY unique_id, ds
"""
logger.info("Fetching training data from %s", BQ_TABLE)
df = client.query(query).to_dataframe()
logger.info("Fetched %d rows", len(df))

if df.empty:
    raise ValueError("No rows returned from BigQuery.")

df["unique_id"] = df["unique_id"].astype(str)
df["ds"] = pd.to_datetime(df["ds"])
df["y"] = pd.to_numeric(df["y"], errors="coerce")
df = df.dropna(subset=["unique_id", "ds", "y"]).sort_values(["unique_id", "ds"])
logger.info("Rows after cleaning: %d", len(df))

counts = df.groupby("unique_id").size()
df = df[df["unique_id"].isin(counts[counts > HORIZON].index)]
logger.info("Rows after short-series filter: %d (series=%d)", len(df), df["unique_id"].nunique())

if df.empty:
    raise ValueError("No series left after filtering short series.")

df.head()

## 2. Train / Validation Split
Hold out the last `HORIZON` steps per series for evaluation.

In [ ]:
train_parts, valid_parts = [], []
for _, g in df.groupby("unique_id", sort=False):
    train_parts.append(g.iloc[:-HORIZON])
    valid_parts.append(g.iloc[-HORIZON:])

train_df = pd.concat(train_parts, ignore_index=True)
valid_df = pd.concat(valid_parts, ignore_index=True)
logger.info("Train rows=%d, validation rows=%d", len(train_df), len(valid_df))
print(f"Train: {len(train_df):,} rows | Validation: {len(valid_df):,} rows")

## 3. Train Each Model and Collect Metrics

In [ ]:
results = {}
all_predictions = []

for model_type in MODEL_TYPES:
    logger.info("Training model_type=%s", model_type)
    model = build_model(model_type)

    fcst = MLForecast(
        models={model_type: model},
        freq=FORECAST_FREQ,
        lags=LAGS,
        date_features=DATE_FEATURES,
        num_threads=1,
    )
    fcst.fit(train_df, id_col="unique_id", time_col="ds", target_col="y")
    preds = fcst.predict(h=HORIZON)

    merged = valid_df.merge(
        preds[["unique_id", "ds", model_type]],
        on=["unique_id", "ds"],
        how="inner",
    ).rename(columns={model_type: "prediction"})

    if merged.empty:
        logger.warning("No overlapping rows for model_type=%s — skipping", model_type)
        continue

    y_true = merged["y"].to_numpy(dtype=float)
    y_pred = merged["prediction"].to_numpy(dtype=float)

    metrics = {
        "rmse":  float(np.sqrt(np.mean((y_true - y_pred) ** 2))),
        "mae":   float(np.mean(np.abs(y_true - y_pred))),
        "wmape": wmape(y_true, y_pred),
    }
    results[model_type] = {"fcst": fcst, "metrics": metrics}
    print(f"{model_type:6s}  rmse={metrics['rmse']:.4f}  mae={metrics['mae']:.4f}  wmape={metrics['wmape']:.4f}")

    row = merged.copy()
    row["run_ts"]             = pd.Timestamp.utcnow()
    row["model_display_name"] = MODEL_DISPLAY_NAME
    row["model_type"]         = model_type
    row["source_table"]       = BQ_TABLE
    row["forecast_freq"]      = FORECAST_FREQ
    row["horizon"]            = HORIZON
    all_predictions.append(row)

if not results:
    raise ValueError("All models failed to produce overlapping predictions.")

## 4. Select Champion

In [ ]:
champion_type = min(
    results,
    key=lambda m: results[m]["metrics"].get(CHAMPION_METRIC, float("inf")),
)
champion_metrics = results[champion_type]["metrics"]
champion_fcst    = results[champion_type]["fcst"]

print(f"Champion: {champion_type}  {CHAMPION_METRIC}={champion_metrics[CHAMPION_METRIC]:.4f}")

## 5. Write Predictions to BigQuery

In [ ]:
preds_df = pd.concat(all_predictions, ignore_index=True)
for col in ["unique_id", "model_display_name", "model_type", "source_table", "forecast_freq"]:
    preds_df[col] = preds_df[col].astype(str)
preds_df["y"]          = pd.to_numeric(preds_df["y"], errors="coerce")
preds_df["prediction"] = pd.to_numeric(preds_df["prediction"], errors="coerce")
preds_df["horizon"]    = preds_df["horizon"].astype("int64")

client.load_table_from_dataframe(
    preds_df[[
        "unique_id", "ds", "y", "prediction", "run_ts",
        "model_display_name", "model_type", "source_table", "forecast_freq", "horizon",
    ]],
    PREDICTION_BQ_TABLE,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_APPEND"),
).result()
logger.info("Wrote %d prediction rows to %s", len(preds_df), PREDICTION_BQ_TABLE)
print(f"Wrote {len(preds_df):,} rows to {PREDICTION_BQ_TABLE}")

## 6. Write Champion Metadata to BigQuery

In [ ]:
champion_row = pd.DataFrame([{
    "model_display_name":  MODEL_DISPLAY_NAME,
    "model_type":          champion_type,
    "source_table":        BQ_TABLE,
    "prediction_bq_table": PREDICTION_BQ_TABLE,
    "forecast_freq":       FORECAST_FREQ,
    "horizon":             HORIZON,
    "lags":                json.dumps(LAGS),
    "date_features":       json.dumps(DATE_FEATURES),
    "series_count":        int(df["unique_id"].nunique()),
    "train_rows":          int(len(train_df)),
    "rmse":                champion_metrics["rmse"],
    "mae":                 champion_metrics["mae"],
    "wmape":               champion_metrics["wmape"],
    "run_ts":              pd.Timestamp.utcnow(),
}])

client.load_table_from_dataframe(
    champion_row,
    CHAMPION_BQ_TABLE,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_APPEND"),
).result()
logger.info("Wrote champion metadata to %s", CHAMPION_BQ_TABLE)
print(f"Champion metadata written to {CHAMPION_BQ_TABLE}")

## 7. Save Champion Model

In [ ]:
model_dir = Path(MODEL_OUTPUT_DIR)
model_dir.mkdir(parents=True, exist_ok=True)
champion_fcst.save(model_dir)

metadata = {
    **champion_metrics,
    "model_type":         champion_type,
    "model_display_name": MODEL_DISPLAY_NAME,
    "champion_metric":    CHAMPION_METRIC,
}
with open(model_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Champion model saved to {model_dir}")
print(json.dumps(metadata, indent=2))

# MLForecast — Vertex AI Pipeline Submission

This notebook submits the MLForecast training pipeline to **Vertex AI Pipelines**.
No training runs locally — all computation happens in the cloud.

### Pipeline modes
| Mode | YAML template | Description |
|------|--------------|-------------|
| `simple` | `nixtla_mlforecast_simple_fanout_pipeline.yaml` | Single model, fast |
| `original` | `nixtla_mlforecast_pipeline.yaml` | Sequential multi-model |
| `fanout` | `nixtla_mlforecast_parallel_pipeline.yaml` | Parallel fan-out per model type |

Set `PIPELINE_MODE` in the **Configuration** cell and run all cells.

In [ ]:
from google.cloud import aiplatform
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("main")

## Configuration
Edit these values before running.

In [ ]:
# ── GCP settings ──────────────────────────────────────────────────────────
PROJECT_ID      = "dazzling-seat-366014"
REGION          = "us-central1"
PIPELINE_ROOT   = "gs://dazzling-seat-366014-vertex-pipelines"
SERVICE_ACCOUNT = "278930531128-compute@developer.gserviceaccount.com"

# ── Pipeline mode: 'simple' | 'original' | 'fanout' ───────────────────────
PIPELINE_MODE = "fanout"

# ── Pipeline parameters ───────────────────────────────────────────────────
PARAMETER_VALUES = {
    "project_id":               PROJECT_ID,
    "bq_table":                f"{PROJECT_ID}.forecasting.sales_daily",
    "prediction_bq_table":     f"{PROJECT_ID}.forecasting.sales_predictions",
    "champion_bq_table":       f"{PROJECT_ID}.forecasting.sales_champion",
    "batch_forecast_bq_table": f"{PROJECT_ID}.forecasting.sales_batch_forecasts",
    "model_types":             ["lgbm", "xgb", "rf", "et", "ridge"],
    "region":                  REGION,
    "endpoint_display_name":   "mlforecast-champion-endpoint",
    "forecast_freq":           "D",
    "horizon":                 2,
    "lags":                    [1, 2, 3],
    "date_features":           ["dayofweek", "month"],
    "model_display_name":      "sales-mlforecast-v3",
    "champion_metric":         "wmape",
}

# ── Template map ──────────────────────────────────────────────────────────
PIPELINE_CONFIG = {
    "simple":   {"template": "nixtla_mlforecast_simple_fanout_pipeline.yaml",  "include_model_types": False, "display_name": "nixtla-mlforecast-simple-run"},
    "original": {"template": "nixtla_mlforecast_pipeline.yaml",                "include_model_types": False, "display_name": "nixtla-mlforecast-original-run"},
    "fanout":   {"template": "nixtla_mlforecast_parallel_pipeline.yaml",        "include_model_types": True,  "display_name": "nixtla-mlforecast-fanout-run"},
}

assert PIPELINE_MODE in PIPELINE_CONFIG, f"Unknown mode '{PIPELINE_MODE}'. Choose: {list(PIPELINE_CONFIG)}"
cfg = PIPELINE_CONFIG[PIPELINE_MODE]
print(f"Mode      : {PIPELINE_MODE}")
print(f"Template  : {cfg['template']}")
print(f"Parameters: {PARAMETER_VALUES}")

## Submit Pipeline to Vertex AI

In [ ]:
parameter_values = dict(PARAMETER_VALUES)
if not cfg["include_model_types"]:
    parameter_values.pop("model_types", None)

aiplatform.init(
    project=PROJECT_ID,
    location=REGION,
    staging_bucket=PIPELINE_ROOT,
)

job = aiplatform.PipelineJob(
    display_name=cfg["display_name"],
    template_path=cfg["template"],
    pipeline_root=PIPELINE_ROOT,
    parameter_values=parameter_values,
)

job.submit(
    service_account=SERVICE_ACCOUNT,
    enable_preflight_validations=True,
)

print(f"Pipeline submitted.")
print(f"Resource name : {job.resource_name}")
print(f"Console URL   : https://console.cloud.google.com/vertex-ai/locations/{REGION}/pipelines/runs?project={PROJECT_ID}")

## Monitor Pipeline
The cell below blocks until the pipeline finishes and prints the final state.
Skip it if you prefer to track progress in the Cloud Console instead.

In [ ]:
job.wait()
print(f"Pipeline finished. State: {job.state}")

## Inspect Champion Results from BigQuery

In [ ]:
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID)
champion_table = PARAMETER_VALUES["champion_bq_table"]

df_champion = client.query(f"""
    SELECT *
    FROM `{champion_table}`
    ORDER BY run_ts DESC
    LIMIT 10
""").to_dataframe()

print(f"Latest champion runs from {champion_table}:")
df_champion